# 05 — Deep Learning : LSTM vs XGBoost
**Savvy Crypto Agent — Jedha Bootcamp Final Project**

Objectif : tester une architecture LSTM sur des séquences de bougies et comparer avec XGBoost.

> **Test set non touché** — on compare les deux modèles sur le même test set que le notebook 04.

## 0. Imports & config

In [ ]:
import sys
!{sys.executable} -m pip install tensorflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import joblib

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

PROCESSED = "C:\\Users\\casir\\Desktop\\CRYPTOS\\data\\processed\\"
MODELS    = "C:\\Users\\casir\\Desktop\\CRYPTOS\\models\\"
os.makedirs(MODELS, exist_ok=True)

# --- latest.json lookup (fallback sans suffixe au premier lancement) ---
if os.path.exists(PROCESSED + 'latest.json'):
    with open(PROCESSED + 'latest.json') as f:
        latest = json.load(f)
    print(f"Chargement version : {latest['date']}")
else:
    latest = {'train': 'train_features.csv', 'test': 'test_features.csv',
              'feature_cols': 'feature_cols.json', 'config': 'config.json', 'date': 'fallback'}
    print('latest.json absent -> fallback sans suffixe')

# --- models_latest.json lookup (fallback sans suffixe) ---
if os.path.exists(MODELS + 'models_latest.json'):
    with open(MODELS + 'models_latest.json') as f:
        models_latest = json.load(f)
    print(f"Modeles version : {models_latest.get('date', '?')}")
else:
    models_latest = {'xgb': 'xgb_final.pkl', 'rf': 'rf_final.pkl',
                     'lstm': 'lstm_final.keras', 'scaler': 'scaler_lstm.pkl',
                     'date': 'fallback'}
    print('models_latest.json absent -> fallback sans suffixe')

with open(PROCESSED + latest['config']) as f:
    cfg = json.load(f)
with open(PROCESSED + 'ml_config.json') as f:
    ml = json.load(f)

TP_NET       = cfg['tp_net']
SL_NET       = cfg['sl_net']
BREAKEVEN    = cfg['breakeven']
FEATURE_COLS = ml['feature_cols']
XGB_THRESH   = ml['xgb_threshold']

WINDOW = 30  # 30 bougies = 2h30 d'historique

print(f'Breakeven : {BREAKEVEN*100:.1f}%')
print(f'Features  : {len(FEATURE_COLS)}')
print(f'Fenetre   : {WINDOW} bougies ({WINDOW*5} minutes)')

## 1. Chargement des données

In [ ]:
train = pd.read_csv(PROCESSED + latest['train'])
test  = pd.read_csv(PROCESSED + latest['test'])
train['candle_time'] = pd.to_datetime(train['candle_time'], utc=True)
test['candle_time']  = pd.to_datetime(test['candle_time'],  utc=True)
train = train.sort_values(['symbol','candle_time']).reset_index(drop=True)
test  = test.sort_values(['symbol','candle_time']).reset_index(drop=True)

# Normalisation des features (important pour LSTM)
from sklearn.preprocessing import StandardScaler
from datetime import datetime
scaler = StandardScaler()
train_scaled = train.copy()
test_scaled  = test.copy()
train_scaled[FEATURE_COLS] = scaler.fit_transform(train[FEATURE_COLS])
test_scaled[FEATURE_COLS]  = scaler.transform(test[FEATURE_COLS])

DATE = datetime.now().strftime('%Y%m%d')
# Versionne
joblib.dump(scaler, MODELS + f'scaler_lstm_{DATE}.pkl')
# Sans suffixe (Railway -- NE PAS SUPPRIMER)
joblib.dump(scaler, MODELS + 'scaler_lstm.pkl')

# Merge dans models_latest.json
models_latest_path = MODELS + 'models_latest.json'
if os.path.exists(models_latest_path):
    with open(models_latest_path) as f:
        models_latest = json.load(f)
else:
    models_latest = {}
models_latest['scaler'] = f'scaler_lstm_{DATE}.pkl'
models_latest['date']   = DATE
with open(models_latest_path, 'w') as f:
    json.dump(models_latest, f, indent=2)

print(f'TRAIN : {len(train):,} | TEST : {len(test):,}')
print(f'Win rate TRAIN : {train["label"].mean()*100:.1f}%')
print(f'Win rate TEST  : {test["label"].mean()*100:.1f}%')
print(f'Scaler -> scaler_lstm_{DATE}.pkl (+ copie sans suffixe)')

## 2. Construction des séquences

Pour chaque bougie t, on crée une séquence de 30 bougies consécutives **du même symbole**.
Le label est celui de la bougie t (le trade hypothétique simulé en notebook 01).

In [ ]:
def create_sequences(df_scaled, feature_cols, window=30):
    X_list, y_list, idx_list = [], [], []
    for sym, group in df_scaled.groupby('symbol'):
        group = group.sort_values('candle_time').reset_index(drop=True)
        feat  = group[feature_cols].values
        labels = group['label'].values
        for i in range(window, len(group)):
            X_list.append(feat[i-window:i])   # (window, n_features)
            y_list.append(labels[i])
            idx_list.append(group.index[i])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

print('Construction des sequences TRAIN...')
X_train_seq, y_train_seq = create_sequences(train_scaled, FEATURE_COLS, WINDOW)
print(f'  X_train : {X_train_seq.shape}  y_train : {y_train_seq.shape}')

print('Construction des sequences TEST...')
X_test_seq, y_test_seq = create_sequences(test_scaled, FEATURE_COLS, WINDOW)
print(f'  X_test  : {X_test_seq.shape}   y_test  : {y_test_seq.shape}')

print(f'Win rate TRAIN seq : {y_train_seq.mean()*100:.1f}%')
print(f'Win rate TEST  seq : {y_test_seq.mean()*100:.1f}%')

# Memoire estimee
mem_gb = (X_train_seq.nbytes + X_test_seq.nbytes) / 1e9
print(f'Memoire sequences  : {mem_gb:.1f} GB')

## 3. Architecture LSTM

In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    print(f'TensorFlow {tf.__version__}')
except ImportError:
    print('TensorFlow non installe. Lance : pip install tensorflow')
    raise

neg = (y_train_seq == 0).sum()
pos = (y_train_seq == 1).sum()
class_weight = {0: 1.0, 1: float(neg/pos)}
print(f'class_weight : {class_weight}')

def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        BatchNormalization(),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )
    return model

lstm_model = build_lstm((WINDOW, len(FEATURE_COLS)))
lstm_model.summary()

## 4. Entraînement LSTM

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3, mode='max', verbose=1)
]

print('Entrainement LSTM...')
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs          = 30,
    batch_size      = 1024,
    validation_split= 0.1,
    class_weight    = class_weight,
    callbacks       = callbacks,
    verbose         = 1
)

# Courbes d'apprentissage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['loss'],     label='Train loss')
ax1.plot(history.history['val_loss'], label='Val loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history.history['auc'],     label='Train AUC')
ax2.plot(history.history['val_auc'], label='Val AUC')
ax2.set_title('AUC')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Comparaison LSTM vs XGBoost sur le TEST SET

In [ ]:
# Probabilites LSTM sur test
print('Calcul des probabilites LSTM sur TEST...')
proba_lstm = lstm_model.predict(X_test_seq, batch_size=2048, verbose=0).flatten()
auc_lstm   = roc_auc_score(y_test_seq, proba_lstm)

# Probabilites XGBoost sur test (subset aligne avec les sequences)
# IMPORTANT : XGBoost recoit les features BRUTES (non normalisees).
# Il a ete entraine sur l'echelle brute au NB 03 ; lui donner test_scaled
# casserait tous ses seuils d'arbres et effondrerait son AUC (~0.81 au lieu de ~0.94).
# Le LSTM, lui, garde bien les sequences normalisees (X_test_seq).
xgb_model = joblib.load(MODELS + models_latest['xgb'])
test_for_xgb = test.groupby('symbol').apply(
    lambda g: g.sort_values('candle_time').iloc[WINDOW:]
).reset_index(drop=True)
proba_xgb_aligned = xgb_model.predict_proba(test_for_xgb[FEATURE_COLS])[:, 1]
auc_xgb_aligned   = roc_auc_score(y_test_seq, proba_xgb_aligned)

# Ensemble : moyenne ponderee
proba_ensemble = 0.5 * proba_lstm + 0.5 * proba_xgb_aligned
auc_ensemble   = roc_auc_score(y_test_seq, proba_ensemble)

print(f'\n=== AUC SUR TEST SET ===')
print(f'  XGBoost  : {auc_xgb_aligned:.4f}')
print(f'  LSTM     : {auc_lstm:.4f}')
print(f'  Ensemble : {auc_ensemble:.4f}')

# Courbes ROC
fig, ax = plt.subplots(figsize=(10, 7))
for name, proba, color in [
    ('XGBoost',  proba_xgb_aligned, '#378ADD'),
    ('LSTM',     proba_lstm,         '#E24B4A'),
    ('Ensemble', proba_ensemble,     '#639922')
]:
    fpr, tpr, _ = roc_curve(y_test_seq, proba)
    auc = roc_auc_score(y_test_seq, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2.5)

ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Aleatoire (AUC=0.50)')
ax.set_title('Courbes ROC — Comparaison XGBoost vs LSTM vs Ensemble (TEST SET)', fontsize=13)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Calibration du seuil pour LSTM et Ensemble

In [ ]:
def threshold_sweep(proba, labels, model_name, color):
    thresholds = np.arange(0.05, 0.98, 0.01)
    results = []
    for t in thresholds:
        mask = proba >= t
        if mask.sum() < 30: break
        wr  = labels[mask].mean()
        n   = mask.sum()
        ev  = wr * TP_NET + (1 - wr) * (-SL_NET)
        results.append({'threshold': round(t,2), 'n': n,
                        'win_rate': round(wr*100,2), 'ev': round(ev*100,3),
                        'rentable': wr > BREAKEVEN})
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f'Calibration seuil — {model_name} (TEST SET)', fontsize=13)
    colors = ['steelblue' if r else 'tomato' for r in df['rentable']]
    axes[0].bar(df['threshold'], df['win_rate'], color=colors, width=0.01)
    axes[0].axhline(BREAKEVEN*100, color='black', linestyle='--',
                    label=f'Breakeven ({BREAKEVEN*100:.1f}%)')
    axes[0].set_ylabel('Win rate (%)')
    axes[0].legend()
    axes[1].bar(df['threshold'], df['n'], color=color, alpha=0.6, width=0.01)
    axes[1].set_ylabel('N trades')
    colors2 = ['steelblue' if e > 0 else 'tomato' for e in df['ev']]
    axes[2].bar(df['threshold'], df['ev'], color=colors2, width=0.01)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].set_ylabel('EV / trade (%)')
    axes[2].set_xlabel('Seuil')
    plt.tight_layout()
    plt.show()

    rentable = df[(df['rentable']) & (df['n'] > 50)]
    if len(rentable):
        best = rentable.loc[rentable['ev'].idxmax()]
        print(f'Meilleur seuil {model_name} : {best["threshold"]}  '
              f'win={best["win_rate"]}%  n={best["n"]:,}  ev={best["ev"]}%')
        return df, best
    print(f'Aucun seuil rentable pour {model_name}')
    return df, None

df_lstm,     best_lstm     = threshold_sweep(proba_lstm,        y_test_seq, 'LSTM',     '#E24B4A')
df_ensemble, best_ensemble = threshold_sweep(proba_ensemble,    y_test_seq, 'Ensemble', '#639922')

## 7. Backtest comparatif XGBoost vs LSTM vs Ensemble

In [ ]:
# Recupere les exit_ret alignes avec les sequences
exit_ret_seq = test_for_xgb['exit_ret'].values

def backtest_seq(proba, thresh, model_name, color, exit_ret, candle_times):
    if thresh is None:
        print(f'Pas de seuil pour {model_name}')
        return
    mask    = proba >= thresh
    rets    = exit_ret[mask]
    labels  = y_test_seq[mask]
    times   = candle_times[mask]

    CAPITAL = 10_000
    SIZE    = 500
    pnl_series = rets * SIZE
    capital    = CAPITAL + np.cumsum(pnl_series)
    drawdown   = capital / np.maximum.accumulate(capital) - 1

    total = len(rets)
    wins  = labels.sum()
    pnl   = pnl_series.sum()
    days  = max((pd.Timestamp(times[-1]) - pd.Timestamp(times[0])).days, 1)

    print(f'=== {model_name} (seuil={thresh}) ===')
    print(f'  Win rate    : {wins/total*100:.1f}%  |  Trades : {total:,}')
    print(f'  PnL total   : {pnl:+,.0f} EUR  ({pnl/CAPITAL*100:+.1f}%)')
    print(f'  Max DD      : {drawdown.min()*100:.2f}%')
    print(f'  PnL/jour    : {pnl/days:+.1f} EUR')
    print()

    fig, (ax1, ax2) = plt.subplots(2,1,figsize=(14,7),sharex=True)
    ax1.plot(capital, color=color, linewidth=1.5)
    ax1.axhline(CAPITAL, color='gray', linestyle=':')
    ax1.fill_between(range(len(capital)), CAPITAL, capital,
                     where=capital>=CAPITAL, alpha=0.15, color=color)
    ax1.fill_between(range(len(capital)), CAPITAL, capital,
                     where=capital<CAPITAL, alpha=0.15, color='tomato')
    ax1.set_title(f'Capital — {model_name} seuil={thresh}', fontsize=13)
    ax1.set_ylabel('Capital (EUR)')
    ax2.fill_between(range(len(drawdown)), drawdown*100, 0, color='tomato', alpha=0.4)
    ax2.set_ylabel('Drawdown (%)')
    plt.tight_layout()
    plt.show()

candle_times_seq = test_for_xgb['candle_time'].values

# Extraction propre des seuils — evite l'ambiguite pandas
thresh_lstm     = float(best_lstm['threshold'])     if best_lstm is not None else None
thresh_ensemble = float(best_ensemble['threshold']) if best_ensemble is not None else None

backtest_seq(proba_xgb_aligned, XGB_THRESH,     'XGBoost',  '#378ADD', exit_ret_seq, candle_times_seq)
backtest_seq(proba_lstm,        thresh_lstm,     'LSTM',     '#E24B4A', exit_ret_seq, candle_times_seq)
backtest_seq(proba_ensemble,    thresh_ensemble, 'Ensemble', '#639922', exit_ret_seq, candle_times_seq)

## 8. Synthèse comparative

In [ ]:
print('=' * 65)
print('SYNTHESE COMPARATIVE -- XGBoost vs LSTM vs Ensemble')
print('=' * 65)
print()
print(f'  AUC XGBoost  : {auc_xgb_aligned:.4f}')
print(f'  AUC LSTM     : {auc_lstm:.4f}')
print(f'  AUC Ensemble : {auc_ensemble:.4f}')
print()

winner = 'XGBoost' if auc_xgb_aligned >= max(auc_lstm, auc_ensemble) else \
         'LSTM'    if auc_lstm >= auc_ensemble else 'Ensemble'
print(f'Meilleur modele par AUC : {winner}')
print()
print('Pourquoi XGBoost performe generalement mieux sur ce type de donnees :')
print('  - Donnees tabulaires avec features bien construites')
print('  - Le feature engineering capture deja la temporalite (ret_1, ret_12, etc.)')
print('  - LSTM apporte de la valeur surtout quand les patterns sont dans la sequence brute')
print('  - Ensemble peut ameliorer si les deux modeles font des erreurs differentes')
print()
print('Pour le Demoday Jedha :')
print('  -> Presenter XGBoost comme modele principal (interprete, rapide, performant)')
print('  -> Presenter LSTM comme exploration deep learning (montre la maitrise du sujet)')
print('  -> Justifier le choix final avec les AUC et les backtests')
print('=' * 65)

from datetime import datetime
DATE = datetime.now().strftime('%Y%m%d')

# Sauvegarde versionnee
lstm_model.save(MODELS + f'lstm_final_{DATE}.keras')
# Sans suffixe (Railway -- NE PAS SUPPRIMER)
lstm_model.save(MODELS + 'lstm_final.keras')

# Merge dans models_latest.json
models_latest_path = MODELS + 'models_latest.json'
if os.path.exists(models_latest_path):
    with open(models_latest_path) as f:
        models_latest = json.load(f)
else:
    models_latest = {}
models_latest['lstm'] = f'lstm_final_{DATE}.keras'
models_latest['date'] = DATE
with open(models_latest_path, 'w') as f:
    json.dump(models_latest, f, indent=2)

# Extraction propre des seuils
thresh_lstm_val     = float(best_lstm['threshold'])     if best_lstm is not None else None
thresh_ensemble_val = float(best_ensemble['threshold']) if best_ensemble is not None else None

# Mise a jour ml_config (versionnee + sans suffixe)
ml['lstm_threshold']     = thresh_lstm_val
ml['ensemble_threshold'] = thresh_ensemble_val
ml['auc_xgb']            = round(float(auc_xgb_aligned), 4)
ml['auc_lstm']           = round(float(auc_lstm), 4)
ml['auc_ensemble']       = round(float(auc_ensemble), 4)
ml['winner_model']       = winner
with open(PROCESSED + f'ml_config_{DATE}.json', 'w') as f:
    json.dump(ml, f, indent=2)
with open(PROCESSED + 'ml_config.json', 'w') as f:
    json.dump(ml, f, indent=2)

print(f'\nlstm_final_{DATE}.keras sauvegarde (+ copie sans suffixe)')
print(f'ml_config_{DATE}.json mis a jour (+ copie sans suffixe)')
print(f'models_latest.json -> {models_latest}')
print('\nNotebook 05 termine.')